In [0]:
import os
import json
import random
import time
import logging
from datetime import datetime, timezone
from faker import Faker
from confluent_kafka import Producer

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

fake = Faker()

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
KAFKA_BROKERS    = os.environ.get("KAFKA_BROKERS",    "pkc-oxqxx9.us-east-1.aws.confluent.cloud:9092")
KAFKA_TOPIC      = os.environ.get("KAFKA_TOPIC",      "topic_ecommerce")
KAFKA_API_KEY    = os.environ.get("KAFKA_API_KEY",    "WGGLYORJWL3ERJAG")
KAFKA_API_SECRET = os.environ.get("KAFKA_API_SECRET", "cflt5yDSkFIQ1oByJHQbMzdvO8EKawMkl1H7xcbIQEf9jfWUU2d+/pa6iq4ZMw2g")
RATE_PER_SEC     = float(os.environ.get("RATE_PER_SEC", "2.0"))   # msgs / second

conf = {
    "bootstrap.servers":  KAFKA_BROKERS,
    "security.protocol":  "SASL_SSL",
    "sasl.mechanism":     "PLAIN",
    "sasl.username":      KAFKA_API_KEY,
    "sasl.password":      KAFKA_API_SECRET,
    "client.id":          "python-faker-producer",
    "compression.type":   "snappy",
}

producer = Producer(conf)

PRODUCTS = [
    "Wireless Mouse", "Mechanical Keyboard", "USB-C Hub", "4K Monitor",
    "Noise Cancelling Headphones", "Webcam 1080p", "Laptop Stand",
    "SSD 1TB", "RAM 32GB", "Gaming Chair", "Smart Watch", "USB Microphone"
]

def delivery_report(err, msg):
    """Callback invoked on message delivery success or failure."""
    if err is not None:
        logger.error(f"Delivery failed: {err}")
    else:
        logger.debug(
            f"Sent to {msg.topic()} partition {msg.partition()} offset {msg.offset()}"
        )

def generate_order() -> dict:
    """
    Generate a single fake order.
    Schema matches the Glue Spark schema exactly:
      order_id:string, customer_id:string, product:string,
      quantity:int, price:double, order_timestamp:timestamp
    """
    return {
        "order_id":        fake.uuid4(),
        "customer_id":     fake.uuid4(),
        "product":         random.choice(PRODUCTS),
        "quantity":        random.randint(1, 5),
        "price":           round(random.uniform(9.99, 299.99), 2),
        "order_timestamp": datetime.now(timezone.utc).isoformat()
    }

def main():
    logger.info(f"Starting producer | Topic: {KAFKA_TOPIC} | Rate: {RATE_PER_SEC} msg/s")
    msg_count = 0
    sleep_interval = 1.0 / RATE_PER_SEC

    try:
        while True:
            order = generate_order()

            producer.produce(
                topic=KAFKA_TOPIC,
                key=order["order_id"],                       # good for partitioning
                value=json.dumps(order, default=str).encode("utf-8"),
                callback=delivery_report
            )

            # Trigger delivery callbacks without blocking
            producer.poll(0)

            msg_count += 1
            if msg_count % 100 == 0:
                logger.info(f"Produced {msg_count} messages...")

            time.sleep(sleep_interval)

    except KeyboardInterrupt:
        logger.info("Shutdown requested.")
    finally:
        logger.info("Flushing remaining messages...")
        producer.flush()
        logger.info(f"Producer stopped. Total produced: {msg_count}")

if __name__ == "__main__":
    main()